# 02 — Feature Engineering

This notebook creates the **canonical model-ready feature tables** for the Moneyball season-wins project.

### Objectives

1. Preserve the original measurements without silently changing their meaning.
2. Add baseball-aware rate and efficiency features using the **appropriate denominator**.
3. Keep season length (`G`) explicit because the primary target is total season wins (`W`).
4. Keep identity fields as metadata only; do not derive franchise/team history from unreliable identity keys.
5. Produce aligned train/prediction feature tables for downstream model experiments.

### Outputs

Written to `data/processed/`:

- `train_fe.csv`
- `pred_fe.csv`

## 1. Design decisions carried forward from 00 and 01

### Observation

The data-quality review found that `ID` is a reliable row identifier, but `teamID`, `franchID`, and `yearID` combinations are not reliable enough to support historical feature construction without additional reconciliation.

EDA also showed that season length varies. This matters because the modelling target is **total wins**, not only win percentage.

### Decisions

- **Do not derive lags, rolling averages, or franchise history in 02.** Identity fields remain metadata only.
- **Keep `G` explicitly as `season_G`.** If a model predicts total wins, it should receive season length directly rather than infer it accidentally from workload totals such as innings pitched.
- **Preserve raw counting statistics unchanged under `raw_...`.** This allows later controlled experiments without losing source information.
- **Create additional domain features with descriptive names.** A suffix such as `_pg`, `_per_AB`, or `_9` states exactly what was normalized.
- **Use the same Pythagorean exponent (`1.83`) as 01 EDA** so the project has one consistent definition.

> **NOTES:** Feature engineering should improve meaning, not obscure it. We keep source totals and engineered rates side-by-side so downstream model experiments can decide what is useful instead of baking an irreversible assumption into 02.

In [ ]:
import numpy as np
import pandas as pd

from moneyball import project_config as cfg
from moneyball import feature_engineering as fe

# ---------------------------------------------------------------------
# Get project configuration
# ---------------------------------------------------------------------

# Apply common notebook display settings and ensure generated 
# project directories exist.
cfg.configure_notebook()
cfg.ensure_project_dirs()


# local aliases
PROJECT_ROOT  = cfg.PROJECT_ROOT
RAW_DIR       = cfg.RAW_DIR
PROCESSED_DIR = cfg.PROCESSED_DIR

TRAIN_PATH    = cfg.TRAIN_PATH
PRED_PATH     = cfg.PRED_PATH
TRAIN_FE_PATH = cfg.TRAIN_FE_PATH
PRED_FE_PATH  = cfg.PRED_FE_PATH

ID_COL        = cfg.ID_COL
YEAR_COL      = cfg.YEAR_COL
TEAM_COL      = cfg.TEAM_COL
FRANCHISE_COL = cfg.FRANCHISE_COL
TARGET_COL    = cfg.TARGET_COL
GAMES_COL     = cfg.GAMES_COL

# Show the resolved paths so it is obvious which files this notebook is using.
cfg.show_project_paths()

Project root : /home/shpang/devs/ntu/projects/baseball_v2
Train        : /home/shpang/devs/ntu/projects/baseball_v2/data/raw/data_year_team_franchise.csv
Prediction   : /home/shpang/devs/ntu/projects/baseball_v2/data/raw/predict_year_team_franchise.csv
Processed    : /home/shpang/devs/ntu/projects/baseball_v2/data/processed


## 2. Load and validate the raw datasets

In [12]:
# ---------------------------------------------------------------------
# Load data files and perform lightweight checks
# ---------------------------------------------------------------------

train_raw = pd.read_csv(TRAIN_PATH)
pred_raw = pd.read_csv(PRED_PATH)

REQUIRED_COMMON = {
    ID_COL, YEAR_COL, TEAM_COL, FRANCHISE_COL, GAMES_COL,
    "R", "RA", "AB", "H", "2B", "3B", "HR", "BB", "SO", "SB",
    "ER", "ERA", "CG", "SHO", "SV", "IPouts", "HA", "HRA",
    "BBA", "SOA", "E", "DP", "FP", "mlb_rpg",
}

missing_train = sorted(REQUIRED_COMMON - set(train_raw.columns))
missing_pred = sorted(REQUIRED_COMMON - set(pred_raw.columns))

assert not missing_train, f"Train is missing required columns: {missing_train}"
assert not missing_pred, f"Prediction is missing required columns: {missing_pred}"
assert TARGET_COL in train_raw.columns, f"Train must contain target column {TARGET_COL!r}."
assert TARGET_COL not in pred_raw.columns, f"Prediction must not contain target column {TARGET_COL!r}."
assert (train_raw[GAMES_COL] > 0).all(), "Train contains non-positive G values."
assert (pred_raw[GAMES_COL] > 0).all(), "Prediction contains non-positive G values."

print("Train shape:", train_raw.shape)
print("Pred shape :", pred_raw.shape)
display(train_raw.head(3))

Train shape: (1812, 52)
Pred shape : (453, 48)


,yearID,teamID,G,R,AB,H,2B,3B,HR,BB,SO,SB,RA,ER,ERA,CG,SHO,SV,IPouts,HA,HRA,BBA,SOA,E,DP,FP,mlb_rpg,era_1,era_2,era_3,era_4,era_5,era_6,era_7,era_8,decade_1910,decade_1920,decade_1930,decade_1940,decade_1950,decade_1960,decade_1970,decade_1980,decade_1990,decade_2000,decade_2010,W,ID,year_label,decade_label,win_bins,franchID
0,1935,BOS,154,718,5288,1458,281,63,69,609,470.0,91,732,619,4.05,82,6,11,4128,1520,67,520,470,190,136.0,0.969,4.864690,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,78,317,1935,1930s,3,BOS
1,1993,TEX,162,835,5510,1472,284,39,181,483,984.0,113,751,684,4.28,20,6,45,4314,1476,144,562,957,130,145.0,0.979,4.597620,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,86,2162,1993,1990s,3,TEX
2,2016,SEA,162,768,5583,1446,251,17,223,506,1288.0,56,707,647,4.00,2,8,49,4371,1410,213,460,1318,89,158.0,0.985,4.477759,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,86,1895,2016,2010s,3,SEA


## 3. Feature treatment policy

The key change from the older 02 notebook is that normalization is **intentional and feature-specific**.

| Source statistic | Treatment in this notebook | Reason |
|---|---|---|
| `G` | Keep as `season_G` | Season length directly affects total possible wins. |
| Raw counting stats | Preserve unchanged as `raw_*` | Do not destroy source information; downstream modelling can test whether totals add value. |
| `R`, `RA` | Add `/G` features | Runs per game measures scoring/run-prevention strength across different season lengths. |
| Batting counts | Derive rates using `AB` or approximate plate appearances | Batting opportunities are more meaningful than blindly dividing by games. |
| Pitching counts | Derive rates using innings pitched | `K/9`, `BB/9`, `H/9`, `HR/9`, and WHIP measure pitching quality independent of workload. |
| `CG`, `SHO`, `SV` | Add `/G` rates | These are game-level outcomes, so games are the natural available denominator. |
| `E`, `DP` | Add `/G` rates | Detailed fielding chances are unavailable; per-game is the most interpretable available exposure adjustment. |
| `ERA`, `FP` | Keep as already-normalized measures | They are already rates/percentages and should not be divided by `G` again. |
| `mlb_rpg` | Keep as league context | Used to adjust scoring and run prevention for era. |
| `era_*`, `decade_*` | Keep as context flags | They describe baseball environment, not accumulated performance. |
| `teamID`, `franchID`, `yearID` | Metadata only | 00 found identity/history ambiguity; no historical aggregates are constructed here. |

> **Decision:** The `raw_*` group is deliberately retained as an experimental source group. It should **not** be interpreted as the recommended final model feature set. The cleaner domain features (`core_`, `bat_`, `pitch_`, `field_`, `ctx_`, `season_`) are designed to be interpretable candidates for later model selection.

## 4. Reusable feature-engineering module

The transformation functions are maintained in:

`src/moneyball/feature_engineering.py`

### Decision

Move repeatable feature-construction logic out of the notebook, while keeping the **observations, design decisions, reasoning, audits, and experiment interpretation** in the notebook.

### Reasoning

- Downstream notebooks should reuse exactly the same feature definitions.
- Feature logic that changes model inputs is easier to test and version in a `.py` module.
- The notebook remains the human-readable record of **why** each feature family exists and what assumptions were made.
- We avoid turning the notebook into a long block of implementation code that must be copied whenever another notebook needs the same transformations.

The module currently contains:

- `safe_divide`
- `add_metadata`
- `add_context_features`
- `add_raw_source_features`
- `add_domain_features`
- `build_feature_table`

> **Human note:** Only reusable transformation logic is extracted. One-off analysis, audits, observations, and modelling decisions remain visible in the notebook.

In [14]:
# ---------------------------------------------------------------------
# Reusable feature functions are imported as `fe`
# ---------------------------------------------------------------------
# Keeping the module namespace explicit makes it easy to see where model-input
# transformations come from when reading the notebook months later.

print("Reusable FE functions:")
for name in [
    "safe_divide",
    "add_metadata",
    "add_context_features",
    "add_raw_source_features",
    "add_domain_features",
    "build_feature_table",
]:
    print(f"  - fe.{name}")

Reusable FE functions:
  - fe.safe_divide
  - fe.add_metadata
  - fe.add_context_features
  - fe.add_raw_source_features
  - fe.add_domain_features
  - fe.build_feature_table


## 5. Baseball-aware engineered features

### 5.1 Core run environment

Runs scored and allowed are naturally compared per game. Run differential combines offense and defense, while league-adjusted versions account for changes in scoring environment.

### 5.2 Batting

Batting rates use at-bats or an **approximate plate-appearance denominator** (`AB + BB`) because HBP, sacrifice flies, etc. are not available in the supplied dataset. The approximation is labelled explicitly in the feature name.

### 5.3 Pitching

Pitching quality is normalized by innings rather than games. This is especially important because total innings pitched is strongly related to season length; using `G` explicitly avoids relying on innings as an accidental schedule-length proxy.

### 5.4 Fielding

`FP` is already a rate. Errors and double plays are converted per game because detailed fielding chances are not available.

In [ ]:
# The implementation of the baseball-aware transformations lives in
# moneyball.feature_engineering.add_domain_features().

print("Domain feature implementation: fe.add_domain_features")

Domain feature implementation: fe.add_domain_features


## 6. Build the canonical train and prediction tables

The build function concatenates metadata, context, untouched raw counts, and engineered feature families. Train receives the two target columns; prediction receives no target information.

> **Decision:** `target_W` remains the primary competition target. `target_W_pct` is kept as an alternative target for later experiments. The choice between predicting wins directly and predicting win percentage belongs in the model notebook, not in feature engineering.

In [16]:
# Build both datasets with the same reusable transformation pipeline.
# Train receives target columns; prediction never receives target information.

train_fe = fe.build_feature_table(train_raw, is_train=True)
pred_fe = fe.build_feature_table(pred_raw, is_train=False)

print("Engineered train shape:", train_fe.shape)
print("Engineered pred shape :", pred_fe.shape)
display(train_fe.head(3))

Engineered train shape: (1812, 78)
Engineered pred shape : (453, 76)


,meta_ID,meta_yearID,meta_teamID,meta_franchID,ctx_mlb_rpg,ctx_era_1,ctx_era_2,ctx_era_3,ctx_era_4,ctx_era_5,ctx_era_6,ctx_era_7,ctx_era_8,ctx_decade_1910,ctx_decade_1920,ctx_decade_1930,ctx_decade_1940,ctx_decade_1950,ctx_decade_1960,ctx_decade_1970,ctx_decade_1980,ctx_decade_1990,ctx_decade_2000,ctx_decade_2010,raw_R,raw_RA,raw_AB,raw_H,raw_2B,raw_3B,raw_HR,raw_BB,raw_SO,raw_SB,raw_ER,raw_CG,raw_SHO,raw_SV,raw_IPouts,raw_HA,raw_HRA,raw_BBA,raw_SOA,raw_E,raw_DP,season_G,core_R_pg,core_RA_pg,core_RDiff_pg,core_Pythag_win_pct,core_R_adj,core_RA_adj,core_RDiff_adj,bat_AVG,bat_OBP_approx,bat_SLG,bat_OPS_approx,bat_ISO,bat_HR_per_AB,bat_BB_rate_approx,bat_SO_rate_approx,bat_XBH_per_H,bat_SB_per_onbase_approx,pitch_ERA,pitch_WHIP,pitch_K_BB,pitch_K9,pitch_BB9,pitch_H9,pitch_HR9,pitch_CG_pg,pitch_SHO_pg,pitch_SV_pg,field_FP,field_E_pg,field_DP_pg,target_W,target_W_pct
0,317,1935,BOS,BOS,4.864690,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,718,732,5288,1458,281,63,69,609,470.0,91,619,82,6,11,4128,1520,67,520,470,190,136.0,154.0,4.662338,4.753247,-0.090909,0.491166,0.958404,0.977091,-0.018688,0.275719,0.350517,0.391831,0.742348,0.116112,0.013048,0.103273,0.079702,0.283265,0.044025,4.05,1.482558,0.903846,3.074128,3.401163,9.941860,0.438227,0.532468,0.038961,0.071429,0.969,1.233766,0.883117,78.0,0.506494
1,2162,1993,TEX,TEX,4.597620,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,835,751,5510,1472,284,39,181,483,984.0,113,684,20,6,45,4314,1476,144,562,957,130,145.0,162.0,5.154321,4.635802,0.518519,0.548355,1.121085,1.008305,0.112780,0.267151,0.326214,0.431397,0.757611,0.164247,0.032849,0.080594,0.164192,0.342391,0.057801,4.28,1.417246,1.702847,5.989569,3.517385,9.237830,0.901252,0.123457,0.037037,0.277778,0.979,0.802469,0.895062,86.0,0.530864
2,1895,2016,SEA,SEA,4.477759,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,768,707,5583,1446,251,17,223,506,1288.0,56,647,2,8,49,4371,1410,213,460,1318,89,158.0,162.0,4.740741,4.364198,0.376543,0.537790,1.058731,0.974639,0.084092,0.259001,0.320578,0.429876,0.750455,0.170876,0.039943,0.083101,0.211529,0.339557,0.028689,4.00,1.283459,2.865217,8.141386,2.841455,8.709677,1.315717,0.012346,0.049383,0.302469,0.985,0.549383,0.975309,86.0,0.530864


## 7. Feature audit and validation

These checks are deliberately strict. If a future edit creates unmatched columns, infinities, unexpected missing values, or target leakage into prediction, the notebook should fail here instead of allowing the problem to reach model training.

In [17]:
# ---------------------------------------------------------------------
# Column alignment
# ---------------------------------------------------------------------
target_cols = [c for c in train_fe.columns if c.startswith("target_")]
train_feature_cols = [c for c in train_fe.columns if c not in target_cols]

missing_in_pred = sorted(set(train_feature_cols) - set(pred_fe.columns))
extra_in_pred = sorted(set(pred_fe.columns) - set(train_feature_cols))

assert not missing_in_pred, f"Prediction is missing feature columns: {missing_in_pred}"
assert not extra_in_pred, f"Prediction has extra feature columns: {extra_in_pred}"
assert not any(c.startswith("target_") for c in pred_fe.columns), "Target leakage into prediction table."

# Put prediction features in exactly the same order as train features.
pred_fe = pred_fe[train_feature_cols]
train_fe = train_fe[train_feature_cols + target_cols]

# ---------------------------------------------------------------------
# Numeric sanity checks
# ---------------------------------------------------------------------
model_numeric_cols = [
    c for c in train_feature_cols
    if not c.startswith("meta_")
]

assert all(pd.api.types.is_numeric_dtype(train_fe[c]) for c in model_numeric_cols), (
    "A non-numeric modelling feature was created."
)

train_numeric = train_fe[model_numeric_cols + target_cols]
pred_numeric = pred_fe[model_numeric_cols]

assert not np.isinf(train_numeric.to_numpy(dtype=float)).any(), "Infinite value in train features."
assert not np.isinf(pred_numeric.to_numpy(dtype=float)).any(), "Infinite value in prediction features."
assert not train_numeric.isna().any().any(), "Missing value in train numeric features."
assert not pred_numeric.isna().any().any(), "Missing value in prediction numeric features."

# Train-only descriptive labels from the source should never enter the feature table.
for forbidden in ["win_bins", "year_label", "decade_label"]:
    assert forbidden not in train_fe.columns
    assert forbidden not in pred_fe.columns

print("✓ Train/pred feature columns align")
print("✓ No target columns in prediction")
print("✓ No infinities or missing numeric values")
print("✓ Train-only descriptive labels excluded")

✓ Train/pred feature columns align
✓ No target columns in prediction
✓ No infinities or missing numeric values
✓ Train-only descriptive labels excluded


In [18]:
# ---------------------------------------------------------------------
# Human-readable feature inventory
# ---------------------------------------------------------------------
feature_groups = []
for prefix, label in [
    ("meta_", "Metadata"),
    ("season_", "Season length"),
    ("ctx_", "Context"),
    ("raw_", "Raw source counts"),
    ("core_", "Core run features"),
    ("bat_", "Batting"),
    ("pitch_", "Pitching"),
    ("field_", "Fielding"),
    ("target_", "Targets"),
]:
    cols = [c for c in train_fe.columns if c.startswith(prefix)]
    feature_groups.append({
        "group": label,
        "count": len(cols),
        "features": ", ".join(cols),
    })

feature_inventory = pd.DataFrame(feature_groups)
display(feature_inventory)

print("\nTrain shape:", train_fe.shape)
print("Pred shape :", pred_fe.shape)

,group,count,features
0,Metadata,4,"meta_ID, meta_yearID, meta_teamID, meta_franchID"
1,Season length,1,season_G
2,Context,20,"ctx_mlb_rpg, ctx_era_1, ctx_era_2, ctx_era_3, ..."
3,Raw source counts,21,"raw_R, raw_RA, raw_AB, raw_H, raw_2B, raw_3B, ..."
4,Core run features,7,"core_R_pg, core_RA_pg, core_RDiff_pg, core_Pyt..."
5,Batting,10,"bat_AVG, bat_OBP_approx, bat_SLG, bat_OPS_appr..."
6,Pitching,10,"pitch_ERA, pitch_WHIP, pitch_K_BB, pitch_K9, p..."
7,Fielding,3,"field_FP, field_E_pg, field_DP_pg"
8,Targets,2,"target_W, target_W_pct"



Train shape: (1812, 78)
Pred shape : (453, 76)


### Observation and future modelling guidance

This notebook now separates three ideas that were mixed together in older versions:

1. **Source information** — `raw_*` preserves original season counts unchanged.
2. **Performance quality** — `core_*`, `bat_*`, `pitch_*`, and `field_*` use baseball-appropriate denominators.
3. **Season opportunity/context** — `season_G` and `ctx_*` describe how much baseball was played and the environment in which it was played.

This separation is intentional. For example, total `IPouts` remains available as a raw source measurement, but pitching quality is represented by innings-normalized features such as `pitch_K9` and `pitch_WHIP`. A later model can test whether the raw workload adds information, but 02 no longer relies on it accidentally as a substitute for `G`.

### Decisions for downstream notebooks

- Treat `meta_*` as identifiers only; exclude them from model matrices.
- Compare feature families explicitly rather than selecting every generated column automatically.
- Start model experiments with interpretable engineered families plus `season_G`/context.
- Test `raw_*` as a separate feature-family experiment, not as an automatic default.
- Keep lag/franchise/cluster features outside this canonical 02 unless a later experiment demonstrates a reliable construction and measurable gain.

## 8. Save canonical processed datasets

The cleaned notebook returns to the simple canonical filenames `train_fe.csv` and `pred_fe.csv`. Older `_v2`, `_lag`, and copied outputs should be treated as historical experiment artifacts during project cleanup.

In [ ]:
train_path = PROCESSED_DIR / "train_fe.csv"
pred_path = PROCESSED_DIR / "pred_fe.csv"

train_fe.to_csv(TRAIN_, index=False)
pred_fe.to_csv(pred_path, index=False)

print("Wrote:", train_path)
print("Wrote:", pred_path)
print("train_fe shape:", train_fe.shape)
print("pred_fe shape :", pred_fe.shape)

Wrote: /home/shpang/devs/ntu/projects/baseball_v2/data/processed/train_fe.csv
Wrote: /home/shpang/devs/ntu/projects/baseball_v2/data/processed/pred_fe.csv
train_fe shape: (1812, 78)
pred_fe shape : (453, 76)


## 9. Final decision record

**Canonical 02 design:**

- Raw counting statistics are preserved unchanged under `raw_*`.
- Rates are added only when the denominator has a clear baseball interpretation.
- `G` is explicit as `season_G` because the primary target is total wins.
- Pitching quality uses innings-based rates rather than total innings as a hidden season-length proxy.
- `ERA` and `FP` are not normalized again because they are already rate statistics.
- `teamID`, `franchID`, and `yearID` are retained for traceability only; no history-derived features are built from them.
- Lag, franchise aggregation, clustering, and other experimental transformations are excluded from canonical feature engineering.
- Both `target_W` and `target_W_pct` are retained so the model notebook can compare target formulations explicitly.